# 00 Setup
Check the environment, run the kit's tests, and see the self-check's to-do list. Everything here runs offline.

In [1]:
import sys, subprocess, importlib.util, pathlib
ROOT = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "src" / "askpanel").exists())
print("Python", sys.version.split()[0], "(3.11 or later)" if sys.version_info >= (3, 11) else "TOO OLD: use 3.11+")
def have(m):
    try:
        return importlib.util.find_spec(m) is not None
    except ModuleNotFoundError:          # e.g. google.genai when google is absent
        return False
for m in ["pandas", "yaml", "pydantic", "mcp", "langgraph", "google.genai", "streamlit"]:
    ok = have(m)
    print(f"  {m:13s} {'ok' if ok else 'missing: pip install -r requirements.txt'}")

Python 3.11.18 (3.11 or later)
  pandas        ok
  yaml          ok
  pydantic      ok
  mcp           ok
  langgraph     missing: pip install -r requirements.txt
  google.genai  missing: pip install -r requirements.txt
  streamlit     missing: pip install -r requirements.txt


In [2]:
r = subprocess.run([sys.executable, "tests/run_tests.py"], cwd=ROOT, capture_output=True, text=True)
print(r.stdout.splitlines()[-1])

26/26 tests pass


In [4]:
import sys, json, pathlib, tempfile
ROOT = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "src" / "askpanel").exists())
sys.path.insert(0, str(ROOT / "src"))
import pandas as pd, yaml
from askpanel import audit, catalog, clock, db, econ, gate, llm, nodes, questions, standin, tools
P = yaml.safe_load(open(ROOT / "params.yaml"))
PANEL = db.load_panel(ROOT / P["data"]["panel"])
TMP = pathlib.Path(tempfile.mkdtemp())
BASE = db.build_db(PANEL, TMP / "base.sqlite")          # tables only: works before you write the view
print(f"{len(PANEL)} panel rows, {PANEL['adsh'].nunique()} filings; scratch database {BASE.name}")

74 panel rows, 4 filings; scratch database base.sqlite


In [5]:
con = gate.connect_ro(BASE)
print(pd.DataFrame(gate.run_query(con, "SELECT * FROM filings")["rows"],
                   columns=gate.run_query(con, "SELECT * FROM filings")["columns"]).to_string(index=False))

                adsh        cik                         name form     period      filed
0009000011-26-000111 9000000011    FIXTURE GOLF CAPITAL CORP 10-Q 2026-06-30 2026-08-10
0009000012-26-000121 9000000012 FIXTURE HOTEL SENIOR LENDING 10-Q 2026-06-30 2026-08-10
0009000013-26-000131 9000000013    FIXTURE INDIA CREDIT FUND 10-Q 2026-06-30 2026-08-10
0009000014-26-000141 9000000014  FIXTURE JULIETT LENDING BDC 10-Q 2026-06-30 2026-08-10


In [6]:
r = subprocess.run([sys.executable, "-m", "askpanel.check"], cwd=ROOT, capture_output=True, text=True,
                   env={**__import__("os").environ, "PYTHONPATH": str(ROOT / "src")})
print(r.stdout)

  PASS  All     required files present             all present
  PASS  Part 0  holdings_v on the fixture          meets the contract
  PASS  Part 0  holdings_v on your real panel      meets the contract
  PASS  Part 1  contract models                    four models and validate_query defined
  PASS  Part 2  catalog conformance                exactly the four READ tools
  PASS  Part 2  read-only server and refusal tests refusal tests present
  PASS  Part 3  no direct database access          agent and app reach data only through the server
  PASS  Part 3  graph shape                        nodes, conditional edges and checkpointer
  PASS  Part 5  question coverage                  ten questions, coverage met
  PASS  Part 5  saved outputs and replay cache     all ten saved and replayable
  PASS  Part 6  unit economics                     five lines, with numbers
  PASS  Part 6  failure log                        5 failure(s) with a filled-in fix (3 required)
12/12 checks pass



**Your turn.** Nothing to build yet. Read the self-check's list above: it is your to-do list for the week. Due **Friday 20 November, 3:59 p.m. ET**.